# Azure ML: Train and Deploy UBI Risk Model

**Run this notebook in Azure ML Studio** (ml.azure.com)

This notebook:
1. Creates a datastore connection to Fabric OneLake
2. Reads training data from Fabric lakehouse Files section
3. Trains the multi-output regression model
4. Registers the model in Azure ML
5. Deploys to a managed online endpoint

**Prerequisites:**
- Run `fabric_prep_training_data.ipynb` in Fabric (all 5 steps) to create and export training data
- Get Lakehouse ID from Fabric: Lakehouse Properties → Copy GUID from ABFS path
- Azure ML workspace managed identity has Contributor/Viewer access to the Fabric workspace

In [1]:
# Install required packages
!pip install azure-ai-ml scikit-learn pandas numpy joblib azure-storage-file-datalake --quiet

In [2]:
# Import libraries
import os
import pandas as pd
import numpy as np
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import (
    Model, 
    ManagedOnlineEndpoint, 
    ManagedOnlineDeployment, 
    CodeConfiguration, 
    Environment,
    OneLakeDatastore,
    OneLakeArtifact
)
from azure.ai.ml.constants import AssetTypes
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import json

## Configuration

In [3]:
# Azure ML Configuration
SUBSCRIPTION_ID = "04054f52-6b7b-47c7-b836-005253626f42"
RESOURCE_GROUP = "RG_ML"
WORKSPACE_NAME = "sbazureml"
MODEL_NAME = "azure_ml_ubi_model"
ENDPOINT_NAME = "ubi-risk-endpoint"

# Fabric OneLake Configuration
DATASTORE_NAME = "fabric_ubi_datastore"
FABRIC_WORKSPACE_ID = "db7dcf85-001e-4277-a85e-3c92029900bc"  # Fabric workspace GUID
FABRIC_LAKEHOUSE_ID = "03795c9e-23af-4272-922d-5f4fec8a2e46"  # TODO: Get from Fabric portal → Lakehouse Properties → ABFS path
ONELAKE_ENDPOINT = "onelake.dfs.fabric.microsoft.com"  # Standard OneLake endpoint
DATA_FILE_PATH = "Files/ml_data/training_data.csv"  # Path within lakehouse Files section (NOT Tables)

print(f"Azure ML workspace: {WORKSPACE_NAME}")
print(f"Model name: {MODEL_NAME}")
print(f"Endpoint name: {ENDPOINT_NAME}")
print(f"OneLake Endpoint: {ONELAKE_ENDPOINT}")
print(f"Fabric Workspace ID: {FABRIC_WORKSPACE_ID}")
print(f"Lakehouse ID: {FABRIC_LAKEHOUSE_ID}")
print(f"Data file path: {DATA_FILE_PATH}")



Azure ML workspace: sbazureml
Model name: azure_ml_ubi_model
Endpoint name: ubi-risk-endpoint
OneLake Endpoint: onelake.dfs.fabric.microsoft.com
Fabric Workspace ID: db7dcf85-001e-4277-a85e-3c92029900bc
Lakehouse ID: 03795c9e-23af-4272-922d-5f4fec8a2e46
Data file path: Files/ml_data/training_data.csv


## Step 1: Connect to Azure ML Workspace

In [4]:
# Authenticate with DefaultAzureCredential (uses Azure ML compute identity)
credential = DefaultAzureCredential()

ml_client = MLClient(
    credential,
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME
)

print(f"✓ Connected to Azure ML workspace: {WORKSPACE_NAME}")

✓ Connected to Azure ML workspace: sbazureml


## Step 2: Create OneLake Datastore

Create a datastore connection to your Fabric lakehouse using identity-based authentication.

In [5]:
# Create OneLake datastore (or get if it already exists)
try:
    # Try to get existing datastore
    fabric_datastore = ml_client.datastores.get(DATASTORE_NAME)
    print(f"✓ Using existing datastore: {DATASTORE_NAME}")
except:
    # Create new datastore
    print(f"Creating new OneLake datastore: {DATASTORE_NAME}...")
    
    fabric_datastore = OneLakeDatastore(
        name=DATASTORE_NAME,
        description="Fabric lakehouse datastore for UBI training data",
        one_lake_workspace_name=FABRIC_WORKSPACE_ID,  # Fabric workspace GUID
        endpoint=ONELAKE_ENDPOINT,  # OneLake endpoint
        artifact=OneLakeArtifact(
            name=f"{FABRIC_LAKEHOUSE_ID}/Files",  # Lakehouse artifact GUID + /Files
            type="lake_house"
        )
    )
    
    ml_client.datastores.create_or_update(fabric_datastore)
    print(f"✓ Datastore created: {DATASTORE_NAME}")

print(f"  Endpoint: {ONELAKE_ENDPOINT}")
print(f"  Workspace GUID: {FABRIC_WORKSPACE_ID}")

print(f"  Artifact: {FABRIC_LAKEHOUSE_ID}/Files")
print(f"  Type: OneLake")
print(f"  Workspace: {FABRIC_WORKSPACE_ID}")

✓ Using existing datastore: fabric_ubi_datastore
  Endpoint: onelake.dfs.fabric.microsoft.com
  Workspace GUID: db7dcf85-001e-4277-a85e-3c92029900bc
  Artifact: 03795c9e-23af-4272-922d-5f4fec8a2e46/Files
  Type: OneLake
  Workspace: db7dcf85-001e-4277-a85e-3c92029900bc


## Step 3: Load Training Data from Datastore

**Important**: Azure ML can only access the **Files** section of a lakehouse, not Tables.

Before running this cell:
1. Export training data to Files in Fabric (run Step 5 in fabric_prep_training_data.ipynb)
2. Get your **Lakehouse ID** (GUID) from Fabric:
   - Open your lakehouse in Fabric
   - Click **Properties**
   - Copy the GUID from the **ABFS path**: `abfss://workspace-guid@onelake.dfs.fabric.microsoft.com/**lakehouse-guid**/Files`
   - Update `FABRIC_LAKEHOUSE_ID` in Configuration cell above

The datastore will read from: `Files/ml_data/training_data.csv`

In [5]:
import mltable
import pandas as pd

# Azure ML datastore parquet file
data_uri = ("azureml://subscriptions/04054f52-6b7b-47c7-b836-005253626f42/resourcegroups/RG_ML/workspaces/sbazureml/datastores/fabric_ubi_datastore/paths/ml_data/training_data.csv")

# Create MLTable
#tbl = mltable.from_parquet_files(paths=[{"file": data_uri}])
tbl = mltable.from_delimited_files(paths=[{"file": data_uri}])

# Load into pandas
df = tbl.to_pandas_dataframe()

df.head()

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration


,policy_number,policy_start_date,policy_end_date,coverage_type,speeding_per_100_miles,harsh_events_per_100_miles,night_miles_share,avg_safety_score,total_trips,total_miles,high_risk_trip_share,baseline_elc,target_risk_factor,target_expected_loss_cost,target_risk_score
0,POL4050,2017-01-02,2018-01-01,Third Party,17.088175,50.170882,0.364101,65.425185,27,731.50,0.0,2172.5,0.000000,0.0,100.0
1,POL2011,2016-07-02,2017-07-01,Third Party,32.124864,114.944330,0.228870,35.519268,41,1276.27,1.0,2172.5,0.000000,0.0,100.0
2,POL4038,2023-01-02,2024-01-01,Third Party,36.252000,136.189947,0.368693,34.331304,23,612.38,1.0,2172.5,0.759494,1650.0,100.0
3,POL2020,2016-08-02,2017-08-01,Third Party,31.650694,106.729085,0.239176,32.491053,19,679.29,1.0,2172.5,0.000000,0.0,100.0
4,POL4020,2025-01-02,2026-01-01,Third Party,12.298907,39.422391,0.258387,65.751923,26,910.65,0.0,2172.5,0.897583,1950.0,100.0


## Step 4: Prepare Features and Targets

In [6]:
# Define feature and target columns
feature_columns = [
    'speeding_per_100_miles',
    'harsh_events_per_100_miles',
    'night_miles_share',
    'avg_safety_score',
    'total_trips',
    'total_miles',
    'high_risk_trip_share',
    'baseline_elc'
]

target_columns = ['target_risk_factor', 'target_expected_loss_cost', 'target_risk_score']

X = df[feature_columns].fillna(0)
y = df[target_columns]

print(f"Features shape: {X.shape}")
print(f"Targets shape: {y.shape}")

Features shape: (82, 8)
Targets shape: (82, 3)


## Step 5: Train Model

In [7]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Train multi-output gradient boosting model
base_model = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

model = MultiOutputRegressor(base_model)
print("\nTraining model...")
model.fit(X_train, y_train)
print("✓ Training complete!")

# Evaluate
y_pred = model.predict(X_test)

print("\n=== Model Performance ===")
for i, col in enumerate(['risk_factor', 'expected_loss_cost', 'risk_score']):
    mse = mean_squared_error(y_test.iloc[:, i], y_pred[:, i])
    r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])
    print(f"{col}:")
    print(f"  MSE: {mse:.4f}")
    print(f"  R²:  {r2:.4f}")

Training set: 65 samples
Test set: 17 samples

Training model...
✓ Training complete!

=== Model Performance ===
risk_factor:
  MSE: 0.4195
  R²:  -0.7525
expected_loss_cost:
  MSE: 1652474.6635
  R²:  -0.5175
risk_score:
  MSE: 8.8670
  R²:  0.9503


## Step 6: Save Model Artifacts

In [8]:
# Create model directory
model_dir = "./ubi_risk_model"
os.makedirs(model_dir, exist_ok=True)

# Save model
model_path = os.path.join(model_dir, "model.pkl")
joblib.dump(model, model_path)
print(f"✓ Model saved to {model_path}")

# Save feature configuration
feature_config = {
    "feature_columns": feature_columns,
    "output_columns": ["risk_factor", "expected_loss_cost", "risk_score"]
}
config_path = os.path.join(model_dir, "feature_config.json")
with open(config_path, "w") as f:
    json.dump(feature_config, f, indent=2)
print(f"✓ Feature config saved to {config_path}")

✓ Model saved to ./ubi_risk_model/model.pkl
✓ Feature config saved to ./ubi_risk_model/feature_config.json


In [34]:
# Create scoring script
score_script = """import json
import joblib
import pandas as pd
import numpy as np
import os
import glob

def init():
    global model, feature_config
    
    # AZUREML_MODEL_DIR points to the model version directory
    model_dir = os.getenv('AZUREML_MODEL_DIR')
    print(f'Model directory: {model_dir}')
    
    # List all files to debug path issues
    for root, dirs, files in os.walk(model_dir):
        print(f'Directory: {root}')
        for file in files:
            print(f'  File: {file}')
    
    # Try to find model.pkl in any subdirectory
    model_pkl_files = glob.glob(os.path.join(model_dir, '**/model.pkl'), recursive=True)
    if model_pkl_files:
        model_path = model_pkl_files[0]
        print(f'Found model at: {model_path}')
    else:
        # Fallback to direct path
        model_path = os.path.join(model_dir, 'model.pkl')
        print(f'Using direct path: {model_path}')
    
    # Find feature_config.json
    config_files = glob.glob(os.path.join(model_dir, '**/feature_config.json'), recursive=True)
    if config_files:
        config_path = config_files[0]
        print(f'Found config at: {config_path}')
    else:
        config_path = os.path.join(model_dir, 'feature_config.json')
        print(f'Using direct config path: {config_path}')
    
    # Load model and config
    model = joblib.load(model_path)
    with open(config_path, 'r') as f:
        feature_config = json.load(f)
    print('Model loaded successfully')

def run(raw_data):
    try:
        data = json.loads(raw_data)
        input_data = data.get('input_data', data)
        df = pd.DataFrame(input_data)
        feature_columns = feature_config['feature_columns']
        X = df[feature_columns].fillna(0)
        predictions = model.predict(X)
        output_columns = feature_config['output_columns']
        result = pd.DataFrame(predictions, columns=output_columns)
        return result.to_dict(orient='records')
    except Exception as e:
        return {'error': str(e)}
"""

score_path = os.path.join(model_dir, "score.py")
with open(score_path, "w") as f:
    f.write(score_script)
print(f"✓ Scoring script saved to {score_path}")


✓ Scoring script saved to ./ubi_risk_model/score.py


In [35]:
# Create conda environment file
conda_content = """name: ubi-risk-scoring
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
    - azureml-defaults==1.38.0
    - inference-schema
    - scikit-learn==1.3.2
    - pandas==2.0.3
    - numpy==1.24.3
    - joblib==1.3.2
"""

conda_path = os.path.join(model_dir, "conda.yaml")
with open(conda_path, "w") as f:
    f.write(conda_content)
print(f"✓ Conda environment saved to {conda_path}")

✓ Conda environment saved to ./ubi_risk_model/conda.yaml


## Step 7: Register Model

In [36]:
# Register the model
model_artifact = Model(
    path=model_dir,
    type=AssetTypes.CUSTOM_MODEL,
    name=MODEL_NAME,
    description="Multi-output UBI risk prediction model: risk_factor, expected_loss_cost, risk_score",
    tags={
        "framework": "scikit-learn",
        "task": "regression",
        "outputs": "risk_factor,expected_loss_cost,risk_score",
        "source": "fabric-lakehouse"
    }
)

print(f"Registering model: {MODEL_NAME}")
registered_model = ml_client.models.create_or_update(model_artifact)
print(f"✓ Model registered: {registered_model.name} version {registered_model.version}")
print(f"✓ Model ID: {registered_model.id}")

# Save model version for deployment
model_version = registered_model.version

# Verify model registration
import time
time.sleep(5)  # Wait for model to propagate
verify_model = ml_client.models.get(name=MODEL_NAME, version=registered_model.version)
print(f"✓ Model verified in workspace: {verify_model.name}:{verify_model.version}")

print(f"\nModel version to deploy: {model_version}")

Registering model: azure_ml_ubi_model
✓ Model registered: azure_ml_ubi_model version 8
✓ Model ID: /subscriptions/04054f52-6b7b-47c7-b836-005253626f42/resourceGroups/RG_ML/providers/Microsoft.MachineLearningServices/workspaces/sbazureml/models/azure_ml_ubi_model/versions/8
✓ Model verified in workspace: azure_ml_ubi_model:8

Model version to deploy: 8


Uploading ubi_risk_model (0.94 MBs): 100%|██████████| 939768/939768 [00:00<00:00, 3801232.35it/s]




## Step 8: Create Managed Online Endpoint

In [37]:
# Create endpoint
endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="UBI risk prediction endpoint",
    auth_mode="key"
)

print(f"Creating endpoint: {ENDPOINT_NAME}...")
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"✓ Endpoint created: {ENDPOINT_NAME}")

Creating endpoint: ubi-risk-endpoint...
✓ Endpoint created: ubi-risk-endpoint


In [38]:
# Just directly reference the curated environment you saw in the UI
# Format: azureml:EnvironmentName@latest (or specific version)

#env_reference = "azureml:AzureML-sklearn-1.5-ubuntu22.04-py310-cpu@latest"
#env_reference = "azureml://registries/azureml/environments/AzureML-sklearn-1.5-ubuntu22.04-py310-cpu/labels/latest"
env_reference = "azureml://registries/azureml/environments/sklearn-1.5/versions/42"

#Creating an environment to perform inference:
custom_env = Environment(
    name="ubi-risk-env",
    description="Custom environment for UBI risk model with sklearn 1.3.2",
    conda_file=os.path.join(model_dir, "conda.yaml"),
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest"
)

In [39]:
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model_artifact,
    code_configuration=CodeConfiguration(
        code=model_dir,
        scoring_script="score.py"
    ),
    environment=env_reference,  # Use the custom-built environment with sklearn 1.3.2
    instance_type="Standard_F4s_v2",
    instance_count=1
)

In [40]:
print(f"Deploying model to endpoint: {ENDPOINT_NAME}... (this may take 5-10 minutes)")
ml_client.online_deployments.begin_create_or_update(deployment).result()
print(f"✓ Deployment complete!")

# Set deployment to receive 100% traffic
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"✓ Traffic routed to deployment 'blue'")

Deploying model to endpoint: ubi-risk-endpoint... (this may take 5-10 minutes)
...........................................................✓ Deployment complete!
✓ Traffic routed to deployment 'blue'


Uploading ubi_risk_model (0.94 MBs): 100%|██████████| 939768/939768 [00:00<00:00, 4441736.73it/s]




## Step 9: Deploy Model to Endpoint

**Note**: Using custom-built environment with exact package versions.

# Get the registered model (from Step 7)
print(f"Retrieving model: {MODEL_NAME}...")
try:
    # Try to get the specific version from Step 7
    deployed_model = ml_client.models.get(name=MODEL_NAME, version=model_version)
    print(f"✓ Found model: {deployed_model.name} v{deployed_model.version}")
except:
    # Fallback: get latest version
    print(f"Using latest version of {MODEL_NAME}")
    deployed_model = ml_client.models.get(name=MODEL_NAME, label="latest")
    print(f"✓ Found model: {deployed_model.name} v{deployed_model.version}")

print(f"Model path: {deployed_model.path}")
print(f"Model ID: {deployed_model.id}")

# Create deployment with custom environment from Step 8.5
print(f"\nUsing custom environment: {env_reference}")

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=deployed_model.id,
    code_configuration=CodeConfiguration(
        code=model_dir,
        scoring_script="score.py"
    ),
    environment=env_reference,  # Use the custom-built environment with sklearn 1.3.2
    instance_type="Standard_DS2_v2",
    instance_count=1
)

print(f"Deploying model to endpoint: {ENDPOINT_NAME}... (this may take 5-10 minutes)")
ml_client.online_deployments.begin_create_or_update(deployment).result()
print(f"✓ Deployment complete!")

# Set deployment to receive 100% traffic
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"✓ Traffic routed to deployment 'blue'")


## Step 10: Test the Endpoint

In [41]:
# Get endpoint details
endpoint_details = ml_client.online_endpoints.get(ENDPOINT_NAME)
scoring_uri = endpoint_details.scoring_uri

# Get primary key
keys = ml_client.online_endpoints.get_keys(ENDPOINT_NAME)
primary_key = keys.primary_key

print("=" * 80)
print("ENDPOINT DETAILS (Save these for deployment-config.ps1):")
print("=" * 80)
print(f"Endpoint URI: {scoring_uri}")
print(f"Primary Key: {primary_key}")
print("=" * 80)

# Test with sample data
test_data = X_test.head(3).to_dict(orient='records')
test_payload = {"input_data": test_data}

import requests
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {primary_key}"
}

response = requests.post(scoring_uri, json=test_payload, headers=headers)

print(f"\n=== Test Predictions ===")
print(response.json())

ENDPOINT DETAILS (Save these for deployment-config.ps1):
Endpoint URI: https://ubi-risk-endpoint.eastus2.inference.ml.azure.com/score
Primary Key: C73Ffb4NEWGBhfzJW2zywiqABonA0ceO2rlZYrR9ZnfRO7v50V4GJQQJ99CCAAAAAAAAAAAAINFRAZML3kWc

=== Test Predictions ===
[{'risk_factor': -0.03489121922811461, 'expected_loss_cost': -4.0994903624643895, 'risk_score': 73.00825594909078}, {'risk_factor': 0.6881463160916059, 'expected_loss_cost': 1997.538375027473, 'risk_score': 99.99970100965642}, {'risk_factor': 1.1269815931740768, 'expected_loss_cost': 2016.253726237549, 'risk_score': 99.99970100965642}]


## Next Steps

**Update your deployment configuration:**

1. Copy the endpoint URI and key from the cell above

2. Update `deployment-config.ps1`:
   ```powershell
   $AzureMLModelName = "azure_ml_ubi_model"
   $AzureMLModelVersion = "<version from cell output above>"
   $AzureMLEndpointName = "ubi-risk-endpoint"
   $AzureMLEndpointUrl = "<scoring_uri from above>"
   $AzureMLEndpointKey = "<primary_key from above>"
   ```

3. Update environment variables in `deploy-containerapp.ps1`

4. Test the production app with ML predictions!